# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"DOI Identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we print the `@id` for each record set and their available fields/columns. All references are made using `@id` as prescribed by the Croissant specification and good practice.


In [ ]:
# List all record sets and their fields (referencing by @id)
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"Fields/columns (@id):")
    for fld in fields:
        # Typically each field can have a '@id' and a 'name'
        name = fld.get('name', '<none>')
        print(f"  - {fld['@id']} ({name})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For this notebook, we select the main protein abundance table record set (`@id`). Adjust the variable below to use other record sets as needed.

In [ ]:
# For demonstration, extract each record set into a DataFrame.

# Collect the list of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"  - Loaded {len(records)} records.")

# For this exploration, pick the first (main) record set.
main_record_set_id = record_set_ids[0]

print(f"\nColumns in DataFrame for RecordSet {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing/collapsing data. This section shows how to filter data, normalize a numeric column, and group by categorical columns.

Replace the field `@id`s as necessary for your dataset below.

In [ ]:
# Identify a numeric field and a group field based on field overview (by @id)
df = dataframes[main_record_set_id]

# Example usage: Find a numeric field (e.g., 'cr:field_coverage_percent') and a group field (e.g., 'cr:field_sample_label')
# List columns
print("DataFrame columns:")
print(df.columns.tolist())

# For this dataset, use likely field @ids for demo
numeric_field_id = None
group_field_id = None

# Try to heuristically pick a numeric and a group field
for col in df.columns:
    if (numeric_field_id is None) and (('percent' in col) or ('abundance' in col.lower()) or ('count' in col)):
        numeric_field_id = col
    elif (group_field_id is None) and ('sample' in col.lower() or 'group' in col.lower() or 'label' in col.lower()):
        group_field_id = col

if numeric_field_id is None:
    numeric_field_id = df.select_dtypes(include='number').columns[0]

print(f"Selected numeric field: {numeric_field_id}")
print(f"Selected group field: {group_field_id if group_field_id else 'N/A'}")

# Ensure the numeric field is numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter out records with the numeric field greater than a threshold
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# If a group field is found, show grouped means
if group_field_id and (group_field_id in filtered_df.columns):
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, plot the distribution of the selected numeric field and, if present, group statistics.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If group field is available, show boxplot
if group_field_id and (group_field_id in filtered_df.columns):
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

- Loaded metadata and records by referencing Croissant `@id`s.
- Inspected record sets and their field structure.
- Extracted and filtered tabular data from the main protein abundance record set.
- Carried out normalization and simple grouping operations for numeric fields.
- Visualized the numeric field's distribution and group-wise comparison.

This template can be extended for deeper biological and statistical analyses tailored to specific research questions.